In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Download CHIRTS-ERA5 daily HI & WBGT JJAS for AP (2000-2020),
clip to bounding box, convert to yearly NetCDFs, and delete GeoTIFFs.

Requires:
    - xarray
    - rioxarray
    - rasterio
    - numpy
    - pandas
    - requests (for simple existence checks)
"""

import os
import shutil
import datetime as dt

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import requests

# ------------------------ CONFIG ---------------------------------

OUT_BASE = "../data/"

# CHIRTS-ERA5 base URLs (check/adjust if needed)
BASE_HI   = "https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily"
BASE_WBGT = "https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/wbgt/tifs/daily"

# Years and months to download
YEARS  = range(2000, 2024)         # 2000–2023
MONTHS = [6, 7, 8, 9]              # JJAS

# Arabian Peninsula bounding box: (lon_min, lon_max, lat_min, lat_max)
AP_BBOX = (30.0, 65.0, 5.0, 35.0)

# Temporary directory for GeoTIFFs
TMP_DIR = os.path.join(OUT_BASE, "tmp_tifs")

os.makedirs(OUT_BASE, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# -----------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------

def build_chirts_url(base, year, date, metric_name):
    # metric_name = "HI" or "WBGT"
    ymd_dotted = date.strftime("%Y.%m.%d")  # YYYY.MM.DD
    return f"{base}/{year}/{metric_name}.{ymd_dotted}.tif"


def url_exists(url, timeout=10):
    """Quick HEAD request to see if file exists."""
    try:
        r = requests.head(url, timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False


def download_tif(url, out_path, chunk_size=2**20):
    """Download a single GeoTIFF via streaming."""
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)


def collect_daily_tifs_for_year(base_url, year, metric_name):
    """
    Download JJAS GeoTIFFs for a given metric & year into TMP_DIR.

    Returns:
        list of (date, filepath) for successfully downloaded files.
    """
    records = []

    for month in MONTHS:
        # Create all days in the month
        start = dt.date(year, month, 1)
        # crude month-end: go to next month, subtract 1 day
        if month == 12:
            end = dt.date(year + 1, 1, 1) - dt.timedelta(days=1)
        else:
            end = dt.date(year, month + 1, 1) - dt.timedelta(days=1)

        current = start
        while current <= end:
            url = build_chirts_url(base_url, year, current, metric_name.upper())
            filename = f"{metric_name.lower()}_{current.strftime('%Y%m%d')}.tif"

            local_path = os.path.join(TMP_DIR, filename)

            if os.path.exists(local_path):
                # Already downloaded earlier
                records.append((current, local_path))
                current += dt.timedelta(days=1)
                continue

            if not url_exists(url):
                print(f"[{metric_name}] Missing on server: {url}")
                current += dt.timedelta(days=1)
                continue

            print(f"[{metric_name}] Downloading {url} -> {local_path}")
            try:
                download_tif(url, local_path)
                records.append((current, local_path))
            except Exception as e:
                print(f"  ERROR downloading {url}: {e}")
                if os.path.exists(local_path):
                    os.remove(local_path)

            current += dt.timedelta(days=1)

    return records


def stack_tifs_to_dataset(records, var_name, bbox):
    """
    Given a list of (date, filepath) and a bounding box,
    open, clip, and stack into an xarray.Dataset time-lat-lon.
    """
    if not records:
        return None

    lons_min, lons_max, lats_min, lats_max = bbox

    data_arrays = []
    times = []

    for date, tif_path in records:
        try:
            da = rioxarray.open_rasterio(tif_path)
            # CHIRTS-ERA5 is usually band=1 for the data
            if "band" in da.dims:
                da = da.sel(band=1, drop=True)

            # Clip to bounding box in the dataset's CRS
            da = da.rio.clip_box(
                minx=lons_min, maxx=lons_max,
                miny=lats_min, maxy=lats_max
            )

            # Standardize dimension names
            # Often da.dims is ('y','x') with coords lat, lon
            # Ensure names are lat/lon; adjust if needed.
            # Many CHIRTS products come with 'lat'/'lon' coords already.
            if "latitude" in da.coords:
                da = da.rename({"latitude": "lat"})
            if "longitude" in da.coords:
                da = da.rename({"longitude": "lon"})
            if "y" in da.dims and "lat" in da.coords:
                da = da.rename({"y": "lat"})
            if "x" in da.dims and "lon" in da.coords:
                da = da.rename({"x": "lon"})

            da = da.squeeze(drop=True)
            da = da.assign_coords(time=pd.Timestamp(date))
            da = da.expand_dims("time")

            da.name = var_name
            data_arrays.append(da)
            times.append(pd.Timestamp(date))
        except Exception as e:
            print(f"  ERROR processing {tif_path}: {e}")

    if not data_arrays:
        return None

    ds = xr.concat(data_arrays, dim="time")
    ds = ds.sortby("time")
    return ds


def save_yearly_netcdf(ds, out_dir, metric_name, year):
    """Save dataset to NetCDF."""
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{metric_name}_AP_{year}.nc")
    print(f"Saving {metric_name} {year} -> {out_path}")
    # Set encoding to keep file sizes manageable
    comp = dict(zlib=True, complevel=4)
    encoding = {metric_name: comp}
    ds.to_netcdf(out_path, encoding=encoding)


def clean_tmp_tifs():
    """Delete all temporary GeoTIFFs."""
    if os.path.isdir(TMP_DIR):
        print(f"Deleting temporary GeoTIFFs in {TMP_DIR}")
        for fname in os.listdir(TMP_DIR):
            path = os.path.join(TMP_DIR, fname)
            if os.path.isfile(path) and path.endswith(".tif"):
                os.remove(path)

# -----------------------------------------------------------------
# Main driver
# -----------------------------------------------------------------

def main():
    for year in YEARS:
        print(f"\n==== YEAR {year} ====")

        # 1) Heat Index
        hi_recs = collect_daily_tifs_for_year(
            base_url=BASE_HI,
            year=year,
            metric_name="hi"
        )
        if hi_recs:
            ds_hi = stack_tifs_to_dataset(hi_recs, var_name="hi", bbox=AP_BBOX)
            if ds_hi is not None and ds_hi.time.size > 0:
                save_yearly_netcdf(ds_hi, os.path.join(OUT_BASE, "hi"), "hi", year)
            else:
                print(f"No valid HI data for {year}")
        else:
            print(f"No HI GeoTIFFs downloaded for {year}")

        # Clean temp tifs before starting WBGT for the year
        clean_tmp_tifs()
        os.makedirs(TMP_DIR, exist_ok=True)

        # 2) WBGT
        wbgt_recs = collect_daily_tifs_for_year(
            base_url=BASE_WBGT,
            year=year,
            metric_name="wbgt"
        )
        if wbgt_recs:
            ds_wbgt = stack_tifs_to_dataset(wbgt_recs, var_name="wbgt", bbox=AP_BBOX)
            if ds_wbgt is not None and ds_wbgt.time.size > 0:
                save_yearly_netcdf(ds_wbgt, os.path.join(OUT_BASE, "wbgt"), "wbgt", year)
            else:
                print(f"No valid WBGT data for {year}")
        else:
            print(f"No WBGT GeoTIFFs downloaded for {year}")

        # Final clean for this year
        clean_tmp_tifs()
        os.makedirs(TMP_DIR, exist_ok=True)


if __name__ == "__main__":
    main()



==== YEAR 2000 ====
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.01.tif -> ../data/tmp_tifs/hi_20000601.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.02.tif -> ../data/tmp_tifs/hi_20000602.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.03.tif -> ../data/tmp_tifs/hi_20000603.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.04.tif -> ../data/tmp_tifs/hi_20000604.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.05.tif -> ../data/tmp_tifs/hi_20000605.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.06.tif -> ../data/tmp_tifs/hi_20000606.tif
[hi] Downloading https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily/2000/HI.2000.06.07.tif -> ../data/tmp_tifs/hi_20000